# Fun

Purpose of this notebook is to actually do training to discover different model 
configurations that would work to train a pretty good TinyGPT. 

In [2]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))


cuda
Tesla T4


In [3]:
from google.colab import drive

drive.mount("/content/drive")


Mounted at /content/drive


In [5]:
%rm -rf 242b-hw3
!git clone https://github.com/sanchitram1/242b-hw3
%cd 242b-hw3

Cloning into '242b-hw3'...
remote: Enumerating objects: 115, done.
remote: Counting objects: 100% (115/115), done.
remote: Compressing objects: 100% (81/81), done.
remote: Total 115 (delta 65), reused 81 (delta 31), pack-reused 0 (from 0)
Receiving objects: 100% (115/115), 2.35 MiB | 199.00 KiB/s, done.
Resolving deltas: 100% (65/65), done.
/content/242b-hw3/242b-hw3


In [6]:
from tokenizer import build_tokenizer
from training import train_model
import json
from utils import save_json, generate_text
from plot import plot_training_curves, plot_validation_curves
from tokenizer import count_tokens, build_token_memmap
from models import TokenChunkDataset, model_checkpoint_path, load_model
from config import (
    DataConfig,
    GlobalTrainingConfig,
    RunConfig,
    TokenConfig,
    TokenizationConfig,
    ModelConfig,
)
from IPython.display import Markdown, Image

In [7]:
token_config = TokenConfig()
data_config = DataConfig()
global_training_config = GlobalTrainingConfig()
tokenization_config = TokenizationConfig()
run_config = RunConfig("experiments-pt4-steps=500_000")

Created /content/drive/MyDrive/courses/242B/HW3/artifacts/runs/experiments-pt4-steps=500_000 and subfolders


In [8]:
tokenizer = build_tokenizer(
    tokenization_config, token_config, data_config.training_file_colab
)

Found it in /content/drive/MyDrive/courses/242B/HW3/artifacts/shared/tokenizers/tinystories_bpe_metaspace_5000_1000000.json, loading from there


In [9]:
train_token_count = count_tokens(
    tokenization_config, tokenizer, data_config.training_file_colab
)
valid_token_count = count_tokens(
    tokenization_config, tokenizer, data_config.validation_file_colab
)
print(f"There are {train_token_count} tokens in train, {valid_token_count} valid")

train_token_memmap_path = build_token_memmap(
    tokenization_config,
    token_config,
    tokenizer,
    data_config.training_file_colab,
    train_token_count,
)
valid_token_memmap_path = build_token_memmap(
    tokenization_config,
    token_config,
    tokenizer,
    data_config.validation_file_colab,
    valid_token_count,
)

There are 176680096 tokens in train, 4850935 valid


In [10]:
train_dataset = TokenChunkDataset(
    train_token_memmap_path, train_token_count, global_training_config.context_length
)
valid_dataset = TokenChunkDataset(
    valid_token_memmap_path, valid_token_count, global_training_config.context_length
)

In [18]:
configs = [
    ModelConfig(
        name="xlarge-plus",
        d_model=768,
        n_heads=12,
        n_layers=6,
        d_ff=3072,
        batch_size=64,
        learning_rate=2.5e-4,
        weight_decay=0.1,
        warmup_steps=1000,
        max_steps=500000,
        use_amp=torch.cuda.is_available(),
    ),
]

In [ ]:
results: dict[str, dict] = {}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for config in configs:
    metrics_path = run_config.metrics / f"{config.name}.json"
    if metrics_path.exists():
        results[config.name] = json.loads(metrics_path.read_text(encoding="utf-8"))
        continue
    result = train_model(
        run_config,
        global_training_config,
        config,
        tokenizer,
        train_dataset,
        valid_dataset,
        device=device,
    )
    results[config.name] = result
    save_json(result, metrics_path)

    if device.type == "cuda":
        torch.cuda.empty_cache()

plot_training_curves(results, run_config.plots / "training_loss_all_models.png")
plot_validation_curves(results, run_config.plots / "validation_loss_all_models.png")

[xlarge-plus] Loss(step=1)=8.66


In [13]:
SAMPLE_PROMPTS = (
    "Once upon a time",
    "A little girl and her pet cat",
    "A fairy woke up",
    "A lion, a bear, and a penguin",
    "The two children",
)

In [6]:
SAMPLE_PROMPTS = (
    "Early one morning",
    "One day, a group of friends went to the park.",
    "Tom and his best friend Brian were playing with their cars.",
    "A cat named Mia wanted to go outside.",
)


In [5]:
# model = load_model(
#     "artifacts/runs/hw3-submission/models/6_xlarge-plus_context=512.pt", "cpu"
# )
model = load_model(run_config.models / "6_xlarge-plus_context=512.pt", "cpu")

In [15]:
generations: list[dict] = []
output_settings = [
    {"temperature": 0.7, "top_k": 30},  # balanced
    {"temperature": 0.5, "top_k": 10},  # conservative
    {"temperature": 0.8, "top_k": 50},  # experimental
]

for prompt in SAMPLE_PROMPTS:
    # for config in configs:
    # model_path = model_checkpoint_path(run_config, config)
    # model = load_model(model_path, device)

    for setting in output_settings:
        temperature = setting["temperature"]
        top_k = setting["top_k"]
        generation = generate_text(
            token_config,
            global_training_config,
            model,
            tokenizer,
            prompt,
            device,
            512,
            temperature,
            top_k,
        )

        generations.append(
            {
                "model": "6_xlarge-plus_context=512",
                "prompt": prompt,
                "generated_text": generation,
                "temperature": temperature,
                "top_k": top_k,
            }
        )

samples_dir = run_config.run_dir / "samples"
samples_dir.mkdir(parents=True, exist_ok=True)

generations_path = samples_dir / "new-big-context-generations.json"
generations_path.write_text(
    json.dumps(generations, indent=2),
    encoding="utf-8",
)

print(f"Saved generations to {generations_path}")


Saved generations to /Users/sanch/Obsidian/2025-spring/Notes/242B/Homeworks/hw3/artifacts/runs/hw3-submission/samples/new-big-context-generations.json


In [15]:
from dataclasses import asdict

manifest = {
    "run_id": run_config.run_id,
    "run_dir": str(run_config.run_dir),
    "device": str(device),
    "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "data": {
        "training_file": str(data_config.training_file),
        "validation_file": str(data_config.validation_file),
    },
    "tokenization": asdict(tokenization_config),
    "global_training": asdict(global_training_config),
    "models": [asdict(config) for config in configs],
    "artifacts": {
        "models_dir": str(run_config.models),
        "metrics_dir": str(run_config.metrics),
        "plots_dir": str(run_config.plots),
        "samples_dir": str(samples_dir),
        "generations": str(generations_path),
    },
    "generation_settings": output_settings,
}

manifest_path = run_config.run_dir / "manifest.json"
manifest_path.write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8",
)

print(f"Saved manifest to {manifest_path}")


Saved manifest to /content/drive/MyDrive/courses/242B/HW3/artifacts/runs/experiments-pt3-context=512/manifest.json


In [16]:
for sample in generations:
    name = sample["model"]
    display(
        Markdown(
            f"**{name}**: Prompt=`{sample['prompt']}` *({sample['temperature']}, {sample['top_k']})*"
        )
    )
    display(Markdown(sample["generated_text"]))


**large-plus**: Prompt=`Once upon a time` *(0.7, 30)*

Once upon a time there was a little girl named Emma. Emma had a special job to mail a letter. She wanted to get a letter to mail the letter and take it to the mail place. Every day she would walk to the mail place and mail it when she got to the mail place. 
One day, Emma was walking and she saw a big tree. She wanted to mail the letter to the mail place. But it wasn't a good idea and she was scared. She thought if she could find a letter and get the letter right.
Emma had an idea. She grabbed some tape and started to put the letter on the tree. The tree was very strong! With the tape and a big smile, Emma got the letter. She was so happy to mail it to the mail place. She mailed the letter to the mailman and gave him the letter. 
Emma was so proud of her letter and she put it in an envelope and went home with her family. She knew she had done a great job mailing the letter and she was so proud of her mail.

**large-plus**: Prompt=`Once upon a time` *(0.5, 10)*

Once upon a time there was a little girl called Lily. She loved to play with her toys and explore the world around her. One day, she decided to explore the world around her. She was brave and curious to see what was on the other side.
Lily found a big tree with a hole in it. She thought it looked like a treasure, so she decided to peek inside. When she opened the hole, she found a beautiful, shiny stone. She was so excited and wanted to take it home.
When she got home, she put the stone in her pocket and went to show her mom the stone. Her mom was so happy to see it and gave her a big hug. Lily felt so proud and proud of herself. She had found something so special and had a wonderful adventure!

**large-plus**: Prompt=`Once upon a time` *(0.8, 50)*

...

The two children were very happy and couldn't wait to show their parents their new discovery.

**xlarge-plus**: Prompt=`The two children` *(0.5, 10)*

The two children were very happy and they laughed together.
But then, a big wind came and blew the two children away. They tried to find their way back, but they could not. They were very sad and scared. They did not know what to do.
Then, a kind bird saw the two children and wanted to help. The bird flew up high and saw the way home. The bird told the children to follow it. The bird led the two children back to their house. The children were very happy and thanked the bird. They learned that when you are kind and help others, good things happen to you too.

**xlarge-plus**: Prompt=`The two children` *(0.8, 50)*

The two children were so excited. They loved playing with blocks and raising them up high. In the end, they had lots of fun and learned an impressive new skill - that if you work hard, you can do anything.